In [ ]:
!pip install timm scikit-learn grad-cam opencv-python matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import timm
from sklearn.metrics import accuracy_score, f1_score
import numpy as np
import copy
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

def get_swin_model(num_classes=11):
    # Load pre-trained Swin-Tiny and configure the classification head
    model = timm.create_model('swin_tiny_patch4_window7_224', pretrained=True, num_classes=num_classes)
    return model.to(device)

In [ ]:
def client_update_feddc(client_model, global_model, dataloader, local_epochs, lr, alpha_dc, local_drift, global_drift):
    client_model.train()
    optimizer = optim.SGD(client_model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    
    global_state = global_model.state_dict()
    
    for epoch in range(local_epochs):
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = client_model(images)
            loss = criterion(outputs, labels)
            
            # --- FedDC Penalty Math ---
            penalty = 0.0
            for name, param in client_model.named_parameters():
                if param.requires_grad:
                    param_diff = param - global_state[name].to(device)
                    drift_diff = local_drift[name].to(device) - global_drift[name].to(device)
                    
                    penalty += torch.sum(param_diff * drift_diff)
                    penalty += (alpha_dc / 2.0) * torch.norm(param_diff) ** 2

            total_loss = loss + penalty
            total_loss.backward()
            optimizer.step()
            
    return client_model.state_dict()

In [ ]:
def client_update_feddc(client_model, global_model, dataloader, local_epochs, lr, alpha_dc, local_drift, global_drift):
    client_model.train()
    optimizer = optim.SGD(client_model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    
    global_state = global_model.state_dict()
    
    for epoch in range(local_epochs):
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = client_model(images)
            loss = criterion(outputs, labels)
            
            # --- FedDC Penalty Math ---
            penalty = 0.0
            for name, param in client_model.named_parameters():
                if param.requires_grad:
                    param_diff = param - global_state[name].to(device)
                    drift_diff = local_drift[name].to(device) - global_drift[name].to(device)
                    
                    penalty += torch.sum(param_diff * drift_diff)
                    penalty += (alpha_dc / 2.0) * torch.norm(param_diff) ** 2

            total_loss = loss + penalty
            total_loss.backward()
            optimizer.step()
            
    return client_model.state_dict()

In [ ]:
def evaluate_model(model, test_loader):
    model.eval()
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for images, labels in test_loader:
            outputs = model(images.to(device))
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    acc = accuracy_score(all_labels, all_preds)
    # Macro F1 is standard for imbalanced medical datasets
    f1 = f1_score(all_labels, all_preds, average='macro') 
    return acc, f1

def train_feddc(global_model, client_loaders, test_loader, num_rounds=50, local_epochs=1, lr=0.01, alpha_dc=0.01):
    num_clients = len(client_loaders)
    
    # Initialize h_i (local drift) and h (global drift) to zeros
    local_drifts = [{name: torch.zeros_like(param) for name, param in global_model.named_parameters() if param.requires_grad} for _ in range(num_clients)]
    global_drift = {name: torch.zeros_like(param) for name, param in global_model.named_parameters() if param.requires_grad}
    
    history = {'acc': [], 'f1': []}

    for round_idx in range(num_rounds):
        local_weights = []
        
        for client_idx in range(num_clients):
            client_model = copy.deepcopy(global_model)
            
            # Local Training
            w_local = client_update_feddc(
                client_model, global_model, client_loaders[client_idx], 
                local_epochs, lr, alpha_dc, local_drifts[client_idx], global_drift
            )
            local_weights.append(copy.deepcopy(w_local))
            
            # Update Local Drift
            global_state = global_model.state_dict()
            with torch.no_grad():
                for name in local_drifts[client_idx].keys():
                    diff = w_local[name].to(device) - global_state[name].to(device)
                    local_drifts[client_idx][name] += alpha_dc * diff
        
        # Server Aggregation (FedAvg logic on the weights)
        global_state = global_model.state_dict()
        for key in global_state.keys():
            global_state[key] = sum([w[key] for w in local_weights]) / num_clients
        global_model.load_state_dict(global_state)
        
        # Update Global Drift
        for name in global_drift.keys():
            global_drift[name] = sum([drift[name] for drift in local_drifts]) / num_clients
            
        # Server Evaluation
        acc, f1 = evaluate_model(global_model, test_loader)
        history['acc'].append(acc)
        history['f1'].append(f1)
        print(f"Round {round_idx+1}/{num_rounds} | Server Acc: {acc:.4f} | Server F1: {f1:.4f}")
            
    return global_model, history

# --- EXECUTION ---
# global_swin = get_swin_model(num_classes=11)
# final_model, stats = train_feddc(global_swin, client_loaders, test_loader, num_rounds=50)

# Uncomment to save your model after training
# torch.save(final_model.state_dict(), 'swin_feddc_milk10k.pth')

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
import cv2

def reshape_transform(tensor, height=7, width=7):
    # Required for Swin Transformers to map the 1D patch sequence back to a 2D image format
    result = tensor.reshape(tensor.size(0), height, width, tensor.size(2))
    result = result.transpose(2, 3).transpose(1, 2)
    return result

def show_grad_cam(model, dataset, image_idx):
    model.eval()
    
    # Grab an image from the test dataset
    image_tensor, label = dataset[image_idx]
    
    # Un-normalize the image tensor so we can display it using Matplotlib
    inv_normalize = transforms.Normalize(
        mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
        std=[1/0.229, 1/0.224, 1/0.225]
    )
    vis_img = inv_normalize(image_tensor).permute(1, 2, 0).numpy()
    vis_img = np.clip(vis_img, 0, 1)

    # Target the last normalization layer of Swin-Tiny
    target_layers = [model.layers[-1].blocks[-1].norm1]
    
    # Generate Heatmap
    cam = GradCAM(model=model, target_layers=target_layers, reshape_transform=reshape_transform)
    targets = [ClassifierOutputTarget(label)]
    
    grayscale_cam = cam(input_tensor=image_tensor.unsqueeze(0).to(device), targets=targets)
    grayscale_cam = grayscale_cam[0, :]
    
    visualization = show_cam_on_image(vis_img, grayscale_cam, use_rgb=True)
    
    # Plotting
    class_names = ['AKIEC', 'BCC', 'BEN_OTH', 'BKL', 'DF', 'INF', 'MAL_OTH', 'MEL', 'NV', 'SCCKA', 'VASC']
    
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(vis_img)
    plt.title(f"Original Scan\n(Ground Truth: {class_names[label]})")
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.imshow(visualization)
    plt.title("Swin-Tiny Grad-CAM Attention")
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()

# --- EXECUTION ---
# You can test different images by changing the image_idx
# show_grad_cam(final_model, test_dataset, image_idx=10)
# show_grad_cam(final_model, test_dataset, image_idx=42)